# 50 Startups 特徵選取分析 (Five Feature Selection Methods)
本筆記本深入解析特徵數量對回歸模型效能 (RMSE 與 $R^2$) 的影響，並實作五種主流特徵選取演算法：
1. **Forward SFS (前向逐步特徵選取)**
2. **LassoCV (L1 正則化稀疏特徵選擇)**
3. **RFECV (遞迴特徵消除 + 交叉驗證)**
4. **ElasticNet Ranking (彈性網絡特徵權重排序)**
5. **Exhaustive Search (最佳子集窮舉搜尋)**

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from itertools import combinations
from sklearn.model_selection import train_test_split, KFold
from sklearn.linear_model import LinearRegression, LassoCV, ElasticNetCV
from sklearn.feature_selection import RFECV, SequentialFeatureSelector
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, r2_score

# 圖表風格設定
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. 資料載入與前處理 (Data Preprocessing)
使用 `One-Hot Encoding` 處理類別變數 `State`，並按照測試集 20% 與 `random_state=0` 進行資料集分割以確保完全重現課堂數據。

In [ ]:
df = pd.read_csv('50_Startups.csv')
df_encoded = pd.get_dummies(df, columns=['State'], dtype=float)

X = df_encoded.drop(columns=['Profit'])
y = df_encoded['Profit']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=0)
print("訓練集維度:", X_train.shape, "測試集維度:", X_test.shape)
print("候選特徵欄位:", list(X.columns))

# 標準化 (用於正則化模型)
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test), columns=X.columns, index=X_test.index)

## 2. 特徵選取方法一：Forward SFS (逐步加入特徵重現圖表數值)
逐步追蹤 1 到 5 個特徵在測試集上的 RMSE 與 $R^2$ 表現。

In [ ]:
feature_progression = [
    ['R&D Spend'],
    ['R&D Spend', 'Marketing Spend'],
    ['R&D Spend', 'Marketing Spend', 'State_New York'],
    ['R&D Spend', 'Marketing Spend', 'State_New York', 'State_Florida'],
    ['R&D Spend', 'Marketing Spend', 'State_New York', 'State_Florida', 'Administration']
]

sfs_results = []
for k, feats in enumerate(feature_progression, 1):
    m = LinearRegression().fit(X_train[feats], y_train)
    preds = m.predict(X_test[feats])
    rmse = np.sqrt(mean_squared_error(y_test, preds))
    r2 = r2_score(y_test, preds)
    sfs_results.append({
        'Number of Features': k,
        'Selected Features': feats,
        'RMSE': rmse,
        'R-squared': r2
    })

df_sfs = pd.DataFrame(sfs_results)
display(df_sfs)

# 繪製目標折線圖
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(df_sfs['Number of Features'], df_sfs['RMSE'], marker='o', color='#1f77b4', lw=2)
axes[0].set_title('RMSE by Number of Features', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Number of Features')
axes[0].set_ylabel('RMSE')

axes[1].plot(df_sfs['Number of Features'], df_sfs['R-squared'], marker='o', color='#1f77b4', lw=2)
axes[1].set_title('R-squared by Number of Features', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Number of Features')
axes[1].set_ylabel('R-squared')
plt.tight_layout()
plt.show()

## 3. 特徵選取方法二：LassoCV (L1 正則化稀疏選擇)
利用 L1 懲罰項將無效特徵係數直接壓縮為 0。

In [ ]:
lasso_cv = LassoCV(cv=5, random_state=0, max_iter=10000).fit(X_train_scaled, y_train)
lasso_coefs = pd.Series(lasso_cv.coef_, index=X.columns)
print(f"LassoCV 最佳 alpha: {lasso_cv.alpha_:.4f}")
print("\n標準化特徵係數:")
print(lasso_coefs.sort_values(ascending=False))

selected_lasso = lasso_coefs[lasso_coefs.abs() > 1e-5].index.tolist()
print(f"\nLassoCV 自動挑選特徵: {selected_lasso}")

m_lasso = LinearRegression().fit(X_train[selected_lasso], y_train)
preds_lasso = m_lasso.predict(X_test[selected_lasso])
print(f"LassoCV 測試集 RMSE: {np.sqrt(mean_squared_error(y_test, preds_lasso)):.6f}")
print(f"LassoCV 測試集 R^2: {r2_score(y_test, preds_lasso):.6f}")

## 4. 特徵選取方法三：RFECV (遞迴特徵消除 + 交叉驗證)
反覆訓練模型並剔除權重最小特徵，透過交叉驗證決定特徵數。

In [ ]:
rfecv = RFECV(
    estimator=LinearRegression(),
    step=1,
    cv=KFold(n_splits=5, shuffle=True, random_state=0),
    scoring='neg_root_mean_squared_error',
    min_features_to_select=1
)
rfecv.fit(X_train, y_train)

print(f"RFECV 推薦最佳特徵數: {rfecv.n_features_}")
print(f"RFECV 選取特徵: {X.columns[rfecv.support_].tolist()}")
ranking_rfecv = pd.Series(rfecv.ranking_, index=X.columns).sort_values()
print("\n特徵排名 (1 為最重要):")
print(ranking_rfecv)

## 5. 特徵選取方法四：ElasticNet Ranking (彈性網絡權重排序)
綜合 L1/L2 懲罰，按絕對係數權重排序特徵重要性。

In [ ]:
enet_cv = ElasticNetCV(l1_ratio=[.1, .5, .7, .9, .95, .99, 1.0], cv=5, random_state=0, max_iter=10000)
enet_cv.fit(X_train_scaled, y_train)
enet_coefs = pd.Series(enet_cv.coef_, index=X.columns)
ranked_enet = enet_coefs.abs().sort_values(ascending=False).index.tolist()

print(f"ElasticNet 最佳 l1_ratio: {enet_cv.l1_ratio_}, alpha: {enet_cv.alpha_:.4f}")
print("\n特徵權重排序:")
for r, f in enumerate(ranked_enet, 1):
    print(f"Rank {r}: {f:<18} (係數: {enet_coefs[f]:.4f})")

## 6. 特徵選取方法五：Exhaustive Search (最佳子集窮舉搜尋)
遍歷每一種特徵數下的所有可能排列組合，求取全局最優解。

In [ ]:
candidate_cols = ['R&D Spend', 'Marketing Spend', 'State_New York', 'State_Florida', 'Administration']
exhaustive_results = []

for k in range(1, len(candidate_cols) + 1):
    best_rmse = float('inf')
    best_comb = None
    best_r2 = None
    for comb in combinations(candidate_cols, k):
        m = LinearRegression().fit(X_train[list(comb)], y_train)
        preds = m.predict(X_test[list(comb)])
        rmse = np.sqrt(mean_squared_error(y_test, preds))
        if rmse < best_rmse:
            best_rmse = rmse
            best_comb = list(comb)
            best_r2 = r2_score(y_test, preds)
    exhaustive_results.append({
        'Number of Features': k,
        'Best Subset': best_comb,
        'RMSE': best_rmse,
        'R-squared': best_r2
    })

df_exhaustive = pd.DataFrame(exhaustive_results)
display(df_exhaustive)